# 01 — Single-Agent Skill Investigation

Alternative to the multi-agent workflow on `main`. **One** `Agent` driven by the
`soc-impossible-travel` file-based Agent Skill (`skills/soc-impossible-travel/SKILL.md`)
performs the entire investigation:

```
Sentinel Detection ──> [Single Agent + soc-impossible-travel Skill]
                          │  (Phase 1) Parse
                          │  (Phase 2) Enrich  ── get_user_context
                          │  (Phase 3) Evaluate 10 risk factors via @tool functions
                          │            (with deterministic scripts: haversine, velocity)
                          │  (Phase 4) Score per scoring_rubric.md
                          └─ (Phase 5) PACO verdict per paco_decision_template.md
                                            │
                                            └─> InvestigationVerdict
```

| Concept                | Usage |
|------------------------|-------|
| `Agent` + `SkillsProvider` | Single agent + file-based skill |
| `@tool`                | Same 10 risk-factor data-source tools as `main` (in `tools.py`) |
| Skill scripts          | `haversine.py`, `travel_velocity.py`, `ip_reputation_lookup.py` |
| Structured output      | `response_format=InvestigationVerdict` |
| Evaluation             | Same 5 test cases as `main` for head-to-head comparison |

> **Prerequisite**: Run `00-setup.ipynb` first.


## Load Config & Imports

In [7]:
import nest_asyncio; nest_asyncio.apply()

import json, time
from pathlib import Path

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

import single_agent_workflow as saw
from single_agent_workflow import (
    NormalizedDetection, InvestigationVerdict,
    build_agent, load_test_case,
)

config_path = Path("workshop_config.json")
if not config_path.exists():
    raise FileNotFoundError("workshop_config.json not found. Run 00-setup.ipynb first.")
with open(config_path) as f:
    config = json.load(f)

PROJECT_ENDPOINT      = config["PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = "gpt-5.4-mini" # config["MODEL_DEPLOYMENT_NAME"]

credential = AzureCliCredential()
af_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=credential,
)
print(f"✅ Connected to {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")


✅ Connected to https://fndryiac2ttkx3.services.ai.azure.com/api/projects/iac-ops-demo
   Model: gpt-5.4-mini


## Inspect the Skill

The agent loads the skill **on demand**. Frontmatter (name + description) is advertised in the system prompt; 
the body of `SKILL.md` is loaded via the `load_skill` tool the first time the agent decides to use it, 
and references / scripts are pulled in via `read_skill_resource` / `run_skill_script`.


In [2]:
skill_dir = Path("skills/soc-impossible-travel")
print("📦 Skill files:")
for p in sorted(skill_dir.rglob("*")):
    if p.is_file():
        print(f"   {p.relative_to(skill_dir.parent)}")


📦 Skill files:
   soc-impossible-travel\references\00_workflow_overview.md
   soc-impossible-travel\references\01_location_anomaly.md
   soc-impossible-travel\references\02_authentication_anomaly.md
   soc-impossible-travel\references\03_token_anomaly.md
   soc-impossible-travel\references\04_brute_force_pattern.md
   soc-impossible-travel\references\05_mfa_abuse.md
   soc-impossible-travel\references\06_post_login_activity.md
   soc-impossible-travel\references\07_oauth_abuse.md
   soc-impossible-travel\references\08_mailbox_manipulation.md
   soc-impossible-travel\references\09_high_privilege_user.md
   soc-impossible-travel\references\10_disabled_account_signin.md
   soc-impossible-travel\references\paco_decision_template.md
   soc-impossible-travel\references\scoring_rubric.md
   soc-impossible-travel\scripts\__pycache__\haversine.cpython-313.pyc
   soc-impossible-travel\scripts\__pycache__\ip_reputation_lookup.cpython-313.pyc
   soc-impossible-travel\scripts\__pycache__\travel_vel

## Build the Single Agent

In [8]:
agent_ctx = build_agent(af_client, store=True)
print("✅ Agent built with file-based skills provider + 11 data-source tools.")


✅ Agent built with file-based skills provider + 11 data-source tools.


## Run a Single Test Case

Change `TEST_CASE` to any of the bundled JSON files (without the `.json` suffix).

In [9]:
TEST_CASE = "01_clear_compromise"

case = load_test_case(Path("test_cases") / f"{TEST_CASE}.json")
detection = NormalizedDetection.model_validate(case["detection"])
print(f"▶ {case['name']}")
print(f"   Expected: {case['expected_verdict']} ({case['expected_confidence']})")
print(f"   UPN:      {detection.PrimaryEntityId}")
print(f"   IoC IPs:  {detection.IoC_IPs}")


▶ Clear Account Compromise
   Expected: AccountCompromised (High)
   UPN:      john.doe@contoso.com
   IoC IPs:  ['203.0.113.42', '198.51.100.7']


In [10]:
async with agent_ctx as agent:
    t0 = time.perf_counter()
    response = await agent.run(detection.model_dump_json())
    elapsed = time.perf_counter() - t0

verdict = InvestigationVerdict.model_validate_json(response.text)
print(f"⏱  {elapsed:.1f}s\n")
print(f"📋 Verdict:    {verdict.verdict}")
print(f"   Confidence: {verdict.confidence}")
print(f"   Reasoning:  {verdict.reasoning}")
print(f"   Actions:    {len(verdict.actionPlan)}")
for a in verdict.actionPlan:
    flag = " ⚠️ approval" if a.requiresApproval else ""
    print(f"     • {a.action} → {a.target} ({a.riskFactor}){flag}")
print(f"\n📝 Narrative:\n{verdict.narrative}")


⏱  17.4s

📋 Verdict:    AccountCompromised
   Confidence: High
   Reasoning:  The investigation shows a malicious sign-in from 198.51.100.7 in Lagos immediately after a clean Seattle session, with MFA bypass, concurrent token activity, and a rapid failed-attempt burst followed by success. Post-login actions confirm active compromise: MFA method tampering, suspicious OAuth consent to SuspiciousApp-DataExfil with Files.ReadWrite.All, and mailbox forwarding/deletion rules; the user is also high-privilege (Exchange Administrator), which raises the impact.
   Actions:    9
     • revokeUserSessions → john.doe@contoso.com (TokenAnomaly)
     • blockSourceIP → 198.51.100.7 (LocationAnomaly)
     • blockSourceIP → 203.0.113.42 (BruteForcePattern)
     • enforcePasswordReset → john.doe@contoso.com (AuthenticationAnomaly)
     • resetMfaMethods → john.doe@contoso.com (MfaAbuse) ⚠️ approval
     • disableUserAccount → john.doe@contoso.com (PostLoginSuspiciousActivity) ⚠️ approval
     • revokeOAu

## Evaluate All 5 Test Cases

Loops the same 5 scenarios used by the multi-agent workflow on `main` so the two approaches can be compared head-to-head 
on verdict accuracy, confidence, and latency.


In [11]:
from datetime import datetime

test_dir = Path("test_cases")
all_cases = sorted(p for p in test_dir.glob("*.json"))
results = []

async with build_agent(af_client, store=True) as eval_agent:
    for case_path in all_cases:
        case = load_test_case(case_path)
        detection = NormalizedDetection.model_validate(case["detection"])
        t0 = time.perf_counter()
        try:
            resp = await eval_agent.run(detection.model_dump_json())
            elapsed = time.perf_counter() - t0
            verdict = InvestigationVerdict.model_validate_json(resp.text)
            actual_verdict, actual_conf, narrative = verdict.verdict, verdict.confidence, verdict.narrative
            action_count = len(verdict.actionPlan)
            err = None
        except Exception as exc:
            elapsed = time.perf_counter() - t0
            actual_verdict, actual_conf, narrative = "ERROR", "ERROR", ""
            action_count = 0
            err = repr(exc)
        results.append({
            "case":                case_path.stem,
            "name":                case["name"],
            "expected_verdict":    case["expected_verdict"],
            "expected_confidence": case["expected_confidence"],
            "actual_verdict":      actual_verdict,
            "actual_confidence":   actual_conf,
            "verdict_match":       actual_verdict == case["expected_verdict"],
            "action_count":        action_count,
            "elapsed_s":           round(elapsed, 2),
            "narrative":           narrative,
            "error":               err,
        })

print(f"{'Case':<32} {'Expected':<22} {'Actual':<22} {'Match':<6} {'Time':>6}")
print("-" * 92)
for r in results:
    exp  = f"{r['expected_verdict']}/{r['expected_confidence']}"
    act  = f"{r['actual_verdict']}/{r['actual_confidence']}"
    mark = "✅" if r["verdict_match"] else "❌"
    print(f"{r['case']:<32} {exp:<22} {act:<22} {mark:<6} {r['elapsed_s']:>5}s")

match_count = sum(1 for r in results if r['verdict_match'])
print(f"\nVerdict accuracy: {match_count}/{len(results)}")


Case                             Expected               Actual                 Match    Time
--------------------------------------------------------------------------------------------
01_clear_compromise              AccountCompromised/High AccountCompromised/High ✅      20.99s
02_benign_business_travel        BenignAnomaly/High     BenignAnomaly/High     ✅      12.48s
03_mfa_fatigue_attack            AccountCompromised/High AccountCompromised/High ✅      16.11s
04_oauth_consent_phishing        AccountCompromised/High AccountCompromised/High ✅      16.86s
05_disabled_account_signin       AccountCompromised/High AccountCompromised/High ✅      15.11s

Verdict accuracy: 5/5


In [12]:
# Save eval results to JSONL for side-by-side comparison with the multi-agent run.
out_dir = Path("eval_results"); out_dir.mkdir(exist_ok=True)
stamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
out_path = out_dir / f"single_agent_skill_{stamp}.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"💾 Saved: {out_path}")


💾 Saved: eval_results\single_agent_skill_20260525T124117Z.jsonl


C:\Users\haozhang2\AppData\Local\Temp\ipykernel_59544\2877734542.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


## (Optional) DevUI

Launch DevUI to interactively chat with the agent and inspect its tool calls + skill loads.


In [ ]:
# from agent_framework_devui import serve
# serve(entities=[build_agent(af_client, store=True)])
